# Download the Ensembl Protein FASTA

In [3]:
! wget https://ftp.ensembl.org/pub/current_fasta/sus_scrofa/pep/Sus_scrofa.Sscrofa11.1.pep.all.fa.gz -O ../data/reference/Sus_scrofa.Sscrofa11.1.pep.all.fa.gz

--2026-08-20 14:58:42--  https://ftp.ensembl.org/pub/current_fasta/sus_scrofa/pep/Sus_scrofa.Sscrofa11.1.pep.all.fa.gz
Resolving ftp.ensembl.org (ftp.ensembl.org)... 193.62.193.169
Connecting to ftp.ensembl.org (ftp.ensembl.org)|193.62.193.169|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10376507 (9.9M) [application/x-gzip]
Saving to: ‘../data/reference/Sus_scrofa.Sscrofa11.1.pep.all.fa.gz’

../data/reference/S 100%[===================>]   9.90M  6.45MB/s    in 1.5s    

2026-08-20 14:58:44 (6.45 MB/s) - ‘../data/reference/Sus_scrofa.Sscrofa11.1.pep.all.fa.gz’ saved [10376507/10376507]



In [2]:
import gzip

filepath = "../data/reference/Sus_scrofa.Sscrofa11.1.pep.all.fa.gz"

with gzip.open(filepath, "rt") as f:
    for _ in range(10):
        line = f.readline()
        if not line:
            break
        print(line, end="")

>ENSSSCP00000010712.5 pep primary_assembly:Sscrofa11.1:14:49001074:49007622:1 gene:ENSSSCG00000010044.5 transcript:ENSSSCT00000010999.5 gene_biotype:IG_C_gene transcript_biotype:IG_C_gene
PKAAPTVNLFPPSSEELGTNKATLVCLISDFYPGAVTVTWKAGGTTVTQGVETTKPSKQS
NNKYAASSYLALSASDWKSSSGFTCQVTHEGTIVEKTVTPSECQPKAAPTVNLFPPSSEE
LGTNKATLVCLISDFYPGAVTVTWKAGGTTVTQGVETTKPSKQSNNKYAASSYLALSASD
WKSSSGFTCQVTHEGTIVEKTVTPSECQPKAAPTVNLFPPSSEELGTNKATLVCLISDFY
PGAVTVTWKAGGTTVTQGVETTKPSKQSNNKYAASSYLALSASDWKSSSGFTCQVTHEGT
IVEKTVTPSECA
>ENSSSCP00000019135.3 pep primary_assembly:Sscrofa11.1:MT:3922:4876:1 gene:ENSSSCG00000018065.4 transcript:ENSSSCT00000019660.4 gene_biotype:protein_coding transcript_biotype:protein_coding gene_symbol:ND1 description:mitochondrially encoded NADH:ubiquinone oxidoreductase core subunit 1 [Source:VGNC Symbol;Acc:VGNC:99793]
MFMINILSLIIPILLAVAFLTLVERKVLGYMQLRKGPNVVGPYGLLQPIADALKLVTKEP
LRPGTSSISMFIIAPILGLSLALTMWVPLPMPYPLINMNLGVLFMLAMSSLAVYSILWSG


In [7]:
import gzip
import pandas as pd
from Bio import SeqIO

# 1. Build a lookup dictionary from the protein FASTA file
fasta_path = "../data/reference/Sus_scrofa.Sscrofa11.1.pep.all.fa.gz"
protein_dict = {}

with gzip.open(fasta_path, "rt") as handle:
    for record in SeqIO.parse(handle, "fasta"):
        # Header contains 'transcript:ENSSSCT...' - extract unversioned ID
        for item in record.description.split():
            if item.startswith("transcript:"):
                tx_id = item.split(":")[1].split(".")[0]
                protein_dict[tx_id] = {
                    "protein_id": record.id.split(".")[0],
                    "seq_len": len(record.seq),
                    "sequence": str(record.seq)
                }

# 2. Load splicing matrix and calculate dIF
df = pd.read_csv("../data/processed/splicing_matrix.csv")
df["dIF"] = df["IF_DMD"] - df["IF_WT"]  # Compute difference in isoform fraction
df["tx_clean"] = df["Transcript_ID"].str.split(".").str[0]

# 3. Map protein metadata
df["protein_id"] = df["tx_clean"].map(lambda x: protein_dict.get(x, {}).get("protein_id"))
df["protein_len"] = df["tx_clean"].map(lambda x: protein_dict.get(x, {}).get("seq_len"))
df["protein_seq"] = df["tx_clean"].map(lambda x: protein_dict.get(x, {}).get("sequence"))

# 4. Inspect top isoform switches with sequence lengths
switches = df[df["dIF"].abs() > 0.2][
    ["Gene_Name", "Gene_ID", "Transcript_ID", "protein_id", "protein_len", "IF_WT", "IF_DMD", "dIF"]
]
print(switches.head())

   Gene_Name             Gene_ID       Transcript_ID          protein_id  \
42      CCNH  ENSSSCG00000014147  ENSSSCT00000015451  ENSSSCP00000015041   
43      CCNH  ENSSSCG00000014147  ENSSSCT00000065980  ENSSSCP00000039263   
44  TMEM161B  ENSSSCG00000014148  ENSSSCT00000015452  ENSSSCP00000015042   
45  TMEM161B  ENSSSCG00000014148  ENSSSCT00000062535  ENSSSCP00000040685   
46     PRR16  ENSSSCG00000039760  ENSSSCT00000050087  ENSSSCP00000051281   

    protein_len     IF_WT    IF_DMD       dIF  
42        303.0  0.391064  0.089223 -0.301841  
43        328.0  0.608934  0.910771  0.301837  
44        398.0  0.999996  0.463092 -0.536904  
45        486.0  0.000000  0.536904  0.536904  
46        304.0  0.000000  0.999952  0.999952  


# Pre-siRNA Processing (Isoform-Specific Target Window Extraction)

In [ ]:
import gzip
import time
import gseapy as gp
import pandas as pd
from Bio import SeqIO

# -----------------------------------------------------------------------------
# 1. EXTRACT ALL GENES IN YOUR TARGET HALLMARK PATHWAYS
# -----------------------------------------------------------------------------
hallmark_library = gp.get_library(name="MSigDB_Hallmark_2020")
keywords = ["E2F", "G2M", "MYOGENESIS", "P53", "APOPTOSIS"]

# Extract every gene symbol belonging to any of your 5 target pathways
pathway_genes = set()
for term, genes in hallmark_library.items():
    if any(kw in term.upper() for kw in keywords):
        pathway_genes.update([g.upper() for g in genes])

print(
    f"[✓] Extracted {len(pathway_genes)} unique genes across target Hallmark"
    " pathways."
)

# -----------------------------------------------------------------------------
# 2. LOAD FASTA PROTEIN DICTIONARY
# -----------------------------------------------------------------------------
fasta_path = "../data/reference/Sus_scrofa.Sscrofa11.1.pep.all.fa.gz"
protein_dict = {}

with gzip.open(fasta_path, "rt") as handle:
    for record in SeqIO.parse(handle, "fasta"):
        for item in record.description.split():
            if item.startswith("transcript:"):
                tx_id = item.split(":")[1].split(".")[0]
                protein_dict[tx_id] = {
                    "protein_id": record.id.split(".")[0],
                    "seq_len": len(record.seq),
                    "sequence": str(record.seq),
                }

# -----------------------------------------------------------------------------
# 3. FILTER SPLICING MATRIX WITH EXTRACTED PATHWAY GENES
# -----------------------------------------------------------------------------
df = pd.read_csv("../data/processed/splicing_matrix.csv")
df["Gene_Name_Clean"] = df["Gene_Name"].dropna().astype(str).str.upper()
df["tx_clean"] = df["Transcript_ID"].str.split(".").str[0]
df["dIF"] = df["IF_DMD"] - df["IF_WT"]

wt_reps = ["SRR32086846", "SRR32086848", "SRR32086850"]
dmd_reps = ["SRR32086830", "SRR32086832", "SRR32086836"]

# Filter 1: Intersect with Pathway Genes
filtered_df = df[df["Gene_Name_Clean"].isin(pathway_genes)].copy()
print(f"Transcripts matching pathway genes: {len(filtered_df)}")

# Filter 2: Isoform shift magnitude (|dIF| >= 0.15) & Abundance (DMD TPM > 1.0)
# (Adjusted TPM threshold to 1.0 to retain low-abundance signaling factors/kinases)
filtered_df = filtered_df[
    (filtered_df["dIF"].abs() >= 0.15) & (filtered_df["mean_TPM_DMD"] > 1.0)
].copy()
print(f"Transcripts passing dIF (>= 0.15) and TPM (> 1.0): {len(filtered_df)}")


# Filter 3: Replicate Directional Consistency
def is_replicate_consistent(row):
    if row["fold_change"] > 1.0:
        return row[dmd_reps].min() > row[wt_reps].max()
    elif row["fold_change"] < 1.0:
        return row[dmd_reps].max() < row[wt_reps].min()
    return False


if not filtered_df.empty:
    filtered_df["consistent"] = filtered_df.apply(
        is_replicate_consistent, axis=1
    )
    filtered_df = filtered_df[filtered_df["consistent"]].copy()
    filtered_df.drop(columns=["consistent"], inplace=True)
print(f"Transcripts passing replicate consistency: {len(filtered_df)}")

# -----------------------------------------------------------------------------
# 4. MAP PROTEIN METADATA & PREDICT NMD / TRUNCATIONS
# -----------------------------------------------------------------------------
filtered_df["protein_id"] = filtered_df["tx_clean"].map(
    lambda x: protein_dict.get(x, {}).get("protein_id")
)
filtered_df["protein_len"] = filtered_df["tx_clean"].map(
    lambda x: protein_dict.get(x, {}).get("seq_len")
)
filtered_df["protein_seq"] = filtered_df["tx_clean"].map(
    lambda x: protein_dict.get(x, {}).get("sequence")
)


def predict_nmd_or_truncation(row):
    if pd.isna(row["protein_len"]) or row["protein_len"] == 0:
        return "Non-coding / NMD Target"
    elif row["protein_len"] < 100:
        return "Severe Truncation (<100 aa)"
    return "Full-Length Candidate"


filtered_df["coding_status"] = filtered_df.apply(
    predict_nmd_or_truncation, axis=1
)

# Display top prioritized target candidates
output_cols = [
    "Gene_Name",
    "Transcript_ID",
    "protein_id",
    "dIF",
    "mean_TPM_DMD",
    "protein_len",
    "coding_status",
]
prioritized_targets = filtered_df[output_cols].sort_values(
    by="dIF", key=abs, ascending=False
)
print("\nTop Candidate Isoform Switches:")
print(prioritized_targets.head(15))     

[✓] Extracted 707 unique genes across target Hallmark pathways.
Transcripts matching pathway genes: 955
Transcripts passing dIF (>= 0.15) and TPM (> 1.0): 94
Transcripts passing replicate consistency: 19

Top Candidate Isoform Switches:
      Gene_Name       Transcript_ID          protein_id       dIF  \
19230       HRC  ENSSSCT00000003506  ENSSSCP00000003423  0.999991   
17030    DLGAP5  ENSSSCT00000005573  ENSSSCP00000005435  0.500432   
452        PKIA  ENSSSCT00000006757  ENSSSCP00000006574  0.436486   
451        PKIA  ENSSSCT00000096901  ENSSSCP00000077394 -0.436485   
16275     CFLAR  ENSSSCT00000060342  ENSSSCP00000037352  0.427087   
11690      MYH8  ENSSSCT00000056000  ENSSSCP00000045044  0.423226   
14858    SH3BGR  ENSSSCT00000078768  ENSSSCP00000072250  0.391429   
16428     ITGB5  ENSSSCT00000062375  ENSSSCP00000056113 -0.365955   
1619      REEP1  ENSSSCT00000029906  ENSSSCP00000021942  0.348382   
1390        PNN  ENSSSCT00000022515  ENSSSCP00000024321 -0.287625   
6607

In [22]:
# 1. Total PKIA isoforms present in your raw dataset (unfiltered)
pkia_all = df[df["Gene_Name_Clean"] == "PKIA"]
print(f"Total raw PKIA isoforms in dataset: {len(pkia_all)}")
print(
    pkia_all[
        ["Transcript_ID", "IF_WT", "IF_DMD", "dIF", "mean_TPM_DMD","mean_TPM_WT"]
    ].to_string()
)

# 2. PKIA isoforms that passed your quality filters (|dIF| >= 0.15, TPM > 1.0, consistency)
pkia_filtered = filtered_df[filtered_df["Gene_Name_Clean"] == "PKIA"]
print(f"PKIA isoforms passing filters: {len(pkia_filtered)}")

Total raw PKIA isoforms in dataset: 2
          Transcript_ID     IF_WT    IF_DMD       dIF  mean_TPM_DMD  mean_TPM_WT
451  ENSSSCT00000096901  0.890323  0.453837 -0.436485     27.289554    11.505562
452  ENSSSCT00000006757  0.109676  0.546162  0.436486     32.841110     1.417340
PKIA isoforms passing filters: 2
